# Restaurant Data Dashboard

This notebook presents an interactive analysis of restaurant data scraped from Google Maps and TripAdvisor for a single city. The dataset includes restaurant names, addresses, categories, scores from two platforms, and user reviews with sentiment analysis.

**Data Source:** [RestaurantsWebScrape](https://github.com/raltomar/RestaurantsWebScrape) — SQLite database with `Restaurants` and `Reviews` tables.

**Audience:** Diners seeking top picks, restaurant owners benchmarking performance, and food critics analyzing trends.

## Table of Contents
1. [Loading the Data](#1-loading-the-data)
2. [Data Validation](#2-data-validation)
3. [Data Preparation](#3-data-preparation)
4. [Visualizations](#4-visualizations)
   - 4.1 Map of Restaurants
   - 4.2 Score Distribution: Google vs TripAdvisor
   - 4.3 Top Cuisine Categories
   - 4.4 Cross-Source Score Comparison
   - 4.5 Review Sentiment vs Rating
   - 4.6 Top 20 Restaurants
5. [Interactive Filters](#5-interactive-filters)
6. [Findings & Notes](#6-findings--notes)

In [28]:
# !pip install folium plotly geopy ipywidgets requests pandas statsmodels

import os
import time
import sqlite3
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import folium
from folium.plugins import MarkerCluster
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

pd.set_option("display.max_columns", None)
print("All imports successful.")

All imports successful.


## 1. Loading the Data

The dataset is a SQLite database scraped from Google Maps and TripAdvisor. We download it once and cache it locally. Two tables are loaded: `Restaurants` (business info + scores) and `Reviews` (individual reviews with sentiment).

In [29]:
DB_URL = "https://github.com/raltomar/RestaurantsWebScrape/raw/refs/heads/main/restaurants_data.db"
DB_PATH = "restaurants_data.db"

def fetch_db(url=DB_URL, path=DB_PATH, force=False):
    if os.path.exists(path) and not force:
        print(f"Database already cached at '{path}'")
        return path
    print(f"Downloading database from {url} ...")
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    size_mb = os.path.getsize(path) / 1e6
    print(f"Saved to '{path}' ({size_mb:.2f} MB)")
    return path

fetch_db()

conn = sqlite3.connect(DB_PATH)
restaurants = pd.read_sql("SELECT * FROM Restaurants", conn)
reviews = pd.read_sql("SELECT * FROM Reviews", conn)
conn.close()

print(f"\nRestaurants: {restaurants.shape[0]} rows x {restaurants.shape[1]} columns")
print(f"Reviews:     {reviews.shape[0]} rows x {reviews.shape[1]} columns")
restaurants.head(3)

Database already cached at 'restaurants_data.db'



Restaurants: 283 rows x 13 columns
Reviews:     944 rows x 6 columns


,id,url,name,address,phone,categories,score,number of reviews,ta score,ta number of reviews,hours,latitude,longitude
0,1,https://www.yellowpages.com/camarillo-ca/mip/p...,Patron Mexican Bar and Grill,"5227 Mission Oaks Blvd, Camarillo, CA 93012",Phone: (805) 621-7251,Restaurants,0.0,0.0,5.0,1.0,Mon - Sun:: 11:00 am - 8:30 pm,34.229024,-118.996447
1,2,https://www.yellowpages.com/ventura-ca/mip/alo...,Aloha Steakhouse,"364 S California St, Ventura, CA 93001",Phone: (805) 652-1799,Steak Houses | Barbecue Restaurants | Bars,4.5,13.0,4.0,454.0,Mon - Sun:: 11:30 am - 10:00 pm,34.276188,-119.293050
2,3,https://www.yellowpages.com/ventura-ca/mip/caf...,Cafe Fiore,"66 S California St, Ventura, CA 93001",Phone: (805) 653-1266,Italian Restaurants | Coffee Shops | Pizza,4.5,5.0,4.0,504.0,Mon - Thu:: 5:00 pm - 10:00 pm | Fri - Sat:: 5...,34.279978,-119.292815


## 2. Data Validation

Before analysis we verify data integrity: null rates, data types, value ranges, and duplicate detection. Cells flagged in red warrant attention.

In [30]:
def validate_table(df, name):
    rows = []
    for col in df.columns:
        series = df[col]
        row = {
            "table": name,
            "column": col,
            "dtype": str(series.dtype),
            "null_count": int(series.isna().sum()),
            "null_pct": round(series.isna().mean() * 100, 1),
            "unique_count": int(series.nunique()),
        }
        if pd.api.types.is_numeric_dtype(series):
            row["min"] = series.min()
            row["max"] = series.max()
        else:
            row["min"] = None
            row["max"] = None
        rows.append(row)
    return pd.DataFrame(rows)

r_val = validate_table(restaurants, "Restaurants")
rv_val = validate_table(reviews, "Reviews")
summary = pd.concat([r_val, rv_val], ignore_index=True)

flags = []
for _, row in summary.iterrows():
    f = []
    if row["null_pct"] > 20:
        f.append("high nulls")
    if row["column"] in ("score", "ta score", "rating") and row["max"] is not None and pd.notna(row["max"]) and row["max"] > 5:
        f.append("score > 5")
    if row["column"] == "sentiment_score" and row["min"] is not None and pd.notna(row["min"]) and (row["min"] < -1 or row["max"] > 1):
        f.append("out of [-1,1]")
    flags.append(", ".join(f))
summary["flags"] = flags

print(f"restaurants['id'] unique: {restaurants['id'].is_unique}")
print(f"restaurants duplicates:   {restaurants.duplicated().sum()}")
print(f"reviews duplicates:       {reviews.duplicated().sum()}\n")

display(summary.style.background_gradient(subset=["null_pct"], cmap="Reds"))

restaurants['id'] unique: True
restaurants duplicates:   0
reviews duplicates:       2



,table,column,dtype,null_count,null_pct,unique_count,min,max,flags
0,Restaurants,id,int64,0,0.000000,283,1.000000,283.000000,
1,Restaurants,url,object,0,0.000000,283,nan,nan,
2,Restaurants,name,object,0,0.000000,276,nan,nan,
3,Restaurants,address,object,1,0.400000,250,nan,nan,
4,Restaurants,phone,object,1,0.400000,269,nan,nan,
5,Restaurants,categories,object,1,0.400000,187,nan,nan,
6,Restaurants,score,float64,2,0.700000,10,0.000000,5.000000,
7,Restaurants,number of reviews,float64,74,26.100000,8,0.000000,13.000000,high nulls
8,Restaurants,ta score,float64,107,37.800000,6,2.500000,5.000000,high nulls
9,Restaurants,ta number of reviews,float64,107,37.800000,87,1.000000,805.000000,high nulls


## 3. Data Preparation

We cast numeric columns, clean strings, explode the comma-separated `categories` field into a long-form DataFrame for analysis, and compute derived columns (`combined_score`, `total_reviews`, `primary_category`).

In [31]:
for col in ["score", "ta score", "number of reviews", "ta number of reviews"]:
    restaurants[col] = pd.to_numeric(restaurants[col], errors="coerce")
for col in ["rating", "sentiment_score"]:
    reviews[col] = pd.to_numeric(reviews[col], errors="coerce")

for col in ["name", "address", "categories", "hours", "phone"]:
    if col in restaurants.columns:
        restaurants[col] = restaurants[col].astype(str).str.strip()
        restaurants[col] = restaurants[col].replace("nan", None)

restaurants["categories_list"] = (
    restaurants["categories"]
    .fillna("")
    .str.split(",")
    .apply(lambda xs: [x.strip() for x in xs if x.strip()])
)

# ── Category clustering ───────────────────────────────────────────────────────
# 193 raw category strings → 12 meaningful clusters, priority-ordered
CATEGORY_CLUSTERS = [
    ("Asian",             ["Sushi", "Japanese", "Chinese", "Thai", "Filipino", "Indian", "Mongolian", "Korean", "Vietnamese", "Asian"]),
    ("Mexican & Latin",   ["Mexican", "Latin American", "Caribbean", "Cuban", "Spanish"]),
    ("Italian & Pizza",   ["Italian", "Pizza"]),
    ("Seafood",           ["Seafood", "Fish & Seafood"]),
    ("Mediterranean",     ["Mediterranean", "Greek", "Middle Eastern", "French"]),
    ("Bars & Nightlife",  ["Cocktail Lounge", "Brew Pub", "Sports Bar", "Tavern", "Night Club", "Wine Bar"]),
    ("Coffee & Bakery",   ["Coffee", "Breakfast", "Brunch", "Bakeries", "Bakery", "Donut", "Bagel", "Cafeteria"]),
    ("Barbecue",          ["Barbecue", "Hawaiian"]),
    ("American",          ["American", "Family Style", "Steak", "Hamburger", "Chicken", "Home Cooking", "Buffet", "Sandwich"]),
    ("Bars",              ["Bar"]),
    ("Desserts",          ["Dessert", "Ice Cream"]),
    ("Health & Specialty",["Health Food", "Juice"]),
]

def assign_cluster(cat_string):
    if not cat_string or (isinstance(cat_string, float) and pd.isna(cat_string)):
        return "Other"
    s = str(cat_string).lower()
    for cluster_name, keywords in CATEGORY_CLUSTERS:
        if any(kw.lower() in s for kw in keywords):
            return cluster_name
    return "Other"

restaurants["cluster"] = restaurants["categories"].apply(assign_cluster)

# Use clustered categories for bar chart and dropdown (one row per restaurant)
categories_long = restaurants[["id", "name", "score", "ta score", "cluster"]].copy()
categories_long = categories_long.rename(columns={"cluster": "category"})

# Derived columns
restaurants["combined_score"] = restaurants[["score", "ta score"]].mean(axis=1)
restaurants["total_reviews"] = (
    restaurants["number of reviews"].fillna(0) + restaurants["ta number of reviews"].fillna(0)
)
restaurants["primary_category"] = restaurants["cluster"]

print(f"Category clusters: {restaurants['cluster'].value_counts().to_dict()}")
restaurants[["name", "score", "ta score", "cluster"]].head()

Category clusters: {'Coffee & Bakery': 49, 'Asian': 44, 'American': 42, 'Mexican & Latin': 35, 'Italian & Pizza': 30, 'Other': 27, 'Bars & Nightlife': 15, 'Barbecue': 13, 'Seafood': 13, 'Mediterranean': 9, 'Health & Specialty': 4, 'Bars': 2}


,name,score,ta score,cluster
0,Patron Mexican Bar and Grill,0.0,5.0,Other
1,Aloha Steakhouse,4.5,4.0,Barbecue
2,Cafe Fiore,4.5,4.0,Italian & Pizza
3,O-sabi Japanese Restaurant,4.5,4.5,Asian
4,The Taj Cafe,4.5,4.5,Asian


### Geocoding Restaurant Addresses

We geocode each restaurant address to latitude/longitude using Nominatim (OpenStreetMap). Results are cached to `geocode_cache.csv` to avoid re-querying. Restaurants that cannot be geocoded are excluded from the map but included in all other charts.

In [32]:
# The database already includes lat/lon coordinates from scraping — use them directly
restaurants = restaurants.rename(columns={"latitude": "lat", "longitude": "lon"})
resolved = restaurants["lat"].notna().sum()
print(f"Coordinates: {resolved}/{len(restaurants)} restaurants have lat/lon from the database.")

Coordinates: 233/283 restaurants have lat/lon from the database.


## 4. Visualizations

All six visualizations are built as reusable factory functions (`make_*`). This design powers both the static renders below and the interactive widget section that follows.

In [33]:
def score_to_color(s):
    if pd.isna(s): return "gray"
    if s >= 4.5: return "green"
    if s >= 4.0: return "lightgreen"
    if s >= 3.5: return "orange"
    return "red"

def make_map(df):
    df_map = df.dropna(subset=["lat", "lon"]).copy()
    if df_map.empty:
        return folium.Map(location=[0, 0], zoom_start=2)
    center = [df_map["lat"].mean(), df_map["lon"].mean()]
    m = folium.Map(location=center, zoom_start=13)
    cluster = MarkerCluster().add_to(m)
    for _, row in df_map.iterrows():
        popup_html = (
            f"<b>{row['name']}</b><br>"
            f"Score: {row.get('score', 'N/A')} | TA: {row.get('ta score', 'N/A')}<br>"
            f"<i>{row.get('categories', 'N/A')}</i><br>"
            f"Hours: {row.get('hours', 'N/A')}<br>"
            f"Phone: {row.get('phone', 'N/A')}"
        )
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=7, fill=True,
            fill_color=score_to_color(row.get("score")),
            color=score_to_color(row.get("score")),
            fill_opacity=0.8,
            popup=folium.Popup(popup_html, max_width=260),
            tooltip=row["name"],
        ).add_to(cluster)
    return m

def make_rating_hist(df):
    fig = go.Figure()
    fig.add_trace(go.Histogram(
        x=df["score"].dropna(), name="Score",
        opacity=0.7, marker_color="#2A9D8F",
        xbins=dict(start=0, end=5, size=0.25),
    ))
    fig.add_trace(go.Histogram(
        x=df["ta score"].dropna(), name="TA Score",
        opacity=0.6, marker_color="#E76F51",
        xbins=dict(start=0, end=5, size=0.25),
    ))
    fig.update_layout(
        barmode="overlay",
        title="Score Distribution: Yellow Pages vs TripAdvisor",
        xaxis_title="Score (0–5)", yaxis_title="Number of Restaurants",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        template="plotly_white",
    )
    fig.add_annotation(
        text=f"n = {df['score'].notna().sum()} restaurants",
        xref="paper", yref="paper", x=0.01, y=0.98,
        showarrow=False, font=dict(size=11, color="gray"),
    )
    return fig

def make_category_bar(df_long, top_n=15):
    counts = df_long["category"].value_counts().head(top_n).reset_index()
    counts.columns = ["category", "count"]
    fig = px.bar(
        counts, x="count", y="category", orientation="h",
        color="count", color_continuous_scale="Teal",
        title=f"Restaurants by Cuisine Cluster",
        labels={"count": "Number of Restaurants", "category": "Cuisine"},
        template="plotly_white",
    )
    fig.update_layout(yaxis=dict(autorange="reversed"), coloraxis_showscale=False)
    return fig

def make_cross_source_scatter(df):
    df_s = df.dropna(subset=["score", "ta score"]).copy()
    if df_s.empty:
        return go.Figure().update_layout(title="No data with both scores")
    fig = px.scatter(
        df_s, x="score", y="ta score", color="primary_category",
        hover_data={"name": True, "score": True, "ta score": True, "primary_category": False},
        title="Score Comparison: Yellow Pages vs TripAdvisor",
        labels={"score": "Score (YP)", "ta score": "TA Score", "primary_category": "Cuisine"},
        template="plotly_white", opacity=0.75,
    )
    corr = df_s["score"].corr(df_s["ta score"])
    fig.add_annotation(
        text=f"Pearson r = {corr:.3f}",
        xref="paper", yref="paper", x=0.02, y=0.95,
        showarrow=False, font=dict(size=12), bgcolor="lightyellow", bordercolor="gray",
    )
    lim = [
        min(df_s["score"].min(), df_s["ta score"].min()) - 0.1,
        max(df_s["score"].max(), df_s["ta score"].max()) + 0.1,
    ]
    fig.add_shape(type="line", x0=lim[0], y0=lim[0], x1=lim[1], y1=lim[1],
                  line=dict(color="gray", dash="dash", width=1))
    return fig

def make_sentiment_scatter(reviews_df):
    df_r = reviews_df.dropna(subset=["sentiment_score", "rating"]).copy()
    if df_r.empty:
        return go.Figure().update_layout(title="No review data")
    try:
        import statsmodels  # noqa: F401
        fig = px.scatter(
            df_r, x="sentiment_score", y="rating", color="source",
            opacity=0.5, trendline="ols",
            title="Review Sentiment Score vs Star Rating",
            labels={"sentiment_score": "Sentiment Score", "rating": "Star Rating"},
            template="plotly_white",
        )
    except ImportError:
        fig = px.scatter(
            df_r, x="sentiment_score", y="rating", color="source",
            opacity=0.5,
            title="Review Sentiment Score vs Star Rating",
            labels={"sentiment_score": "Sentiment Score", "rating": "Star Rating"},
            template="plotly_white",
        )
    fig.update_layout(
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    return fig

def make_top_table(df, n=20):
    cols = ["name", "score", "ta score", "number of reviews", "categories"]
    df_top = df.sort_values("score", ascending=False).head(n)[cols].reset_index(drop=True)
    n_rows = len(df_top)
    row_colors = ["#edf6f4" if i % 2 == 0 else "#ffffff" for i in range(n_rows)]
    col_fill = [row_colors] * len(cols)
    fig = go.Figure(data=[go.Table(
        columnwidth=[25, 180, 70, 70, 80, 280],
        header=dict(
            values=["#", "Name", "Score", "TA Score", "# Reviews", "Categories"],
            fill_color="#264653",
            font=dict(color="#ffffff", size=12, family="Arial"),
            align="left",
            height=36,
        ),
        cells=dict(
            values=[
                list(range(1, n_rows + 1)),
                df_top["name"].tolist(),
                df_top["score"].round(2).tolist(),
                df_top["ta score"].round(2).tolist(),
                df_top["number of reviews"].fillna(0).astype(int).tolist(),
                df_top["categories"].tolist(),
            ],
            fill_color=col_fill,
            align="left",
            font=dict(color="#1a1a1a", size=11, family="Arial"),
            height=30,
        ),
    )])
    fig.update_layout(
        title=f"Top {n} Restaurants by Score",
        template="plotly_white",
        margin=dict(l=10, r=10, t=50, b=10),
    )
    return fig

print("Chart factory functions defined.")

Chart factory functions defined.


### 4.1 Map of Restaurants

Each restaurant is plotted as a circle marker colored by its Google score: green (≥4.5), light green (≥4.0), orange (≥3.5), red (<3.5). Click any cluster to zoom in, and click a marker to see restaurant details.

In [34]:
make_map(restaurants)

### 4.2 Score Distribution: Google vs TripAdvisor

Overlaid histograms show how scores are distributed across both platforms. Divergence between the two distributions indicates systematic scoring differences between Google and TripAdvisor reviewers.

In [35]:
make_rating_hist(restaurants).show()

### 4.3 Top Cuisine Categories

The most common cuisine types in the dataset. Categories are extracted by splitting the comma-separated `categories` field, so one restaurant may appear in multiple bars.

In [36]:
make_category_bar(categories_long).show()

### 4.4 Cross-Source Score Comparison

Each point represents one restaurant plotted by its Google score (x-axis) vs TripAdvisor score (y-axis). The dashed diagonal is the line of perfect agreement. Points above the line are rated higher on TripAdvisor; below on Google. The Pearson correlation coefficient quantifies overall agreement.

In [37]:
make_cross_source_scatter(restaurants).show()

### 4.5 Review Sentiment vs Star Rating

Does the emotional tone of a review text correlate with the numeric rating given? This scatter plots NLP-derived sentiment score against the star rating for every review, colored by source platform. An OLS trendline confirms the direction and strength of the relationship.

In [38]:
make_sentiment_scatter(reviews).show()

### 4.6 Top 20 Restaurants

The highest-rated restaurants by Google score, including their TripAdvisor score, total review count, and cuisine categories.

In [39]:
make_top_table(restaurants).show()

## 5. Interactive Filters

Use the controls below to slice the data. All six visualizations update in real time. Interactive elements:
1. **Cuisine dropdown** — filter to a specific cuisine type
2. **Min score slider** — show only restaurants above a score threshold
3. **Min reviews slider** — exclude restaurants with few reviews
4. **Sources selector** — choose which review platforms to include in the sentiment chart
5. **Top N slider** — control how many categories appear in the bar chart
6. **Show map checkbox** — toggle the (slower) map render on/off

In [40]:
all_categories = sorted(restaurants["cluster"].dropna().unique().tolist())
all_sources = sorted(reviews["source"].dropna().unique().tolist())
max_reviews = int(restaurants["number of reviews"].max() or 0)

cuisine_dd = widgets.Dropdown(
    options=["(All)"] + all_categories, value="(All)",
    description="Cuisine:", style={"description_width": "initial"},
)
min_rating = widgets.FloatSlider(
    min=0.0, max=5.0, step=0.1, value=0.0,
    description="Min score:", style={"description_width": "initial"},
    layout=widgets.Layout(width="380px"),
)
min_reviews_w = widgets.IntSlider(
    min=0, max=max_reviews, value=0,
    description="Min reviews:", style={"description_width": "initial"},
    layout=widgets.Layout(width="380px"),
)
source_select = widgets.SelectMultiple(
    options=all_sources, value=tuple(all_sources),
    description="Sources:", style={"description_width": "initial"},
    rows=min(4, max(1, len(all_sources))),
)
top_n_slider = widgets.IntSlider(
    min=5, max=30, value=15,
    description="Top N cats:", style={"description_width": "initial"},
    layout=widgets.Layout(width="380px"),
)
show_map_cb = widgets.Checkbox(value=True, description="Show map")

out = widgets.Output()

def filter_restaurants(df, cuisine, min_score, min_revs):
    mask = pd.Series([True] * len(df), index=df.index)
    if cuisine != "(All)":
        mask &= df["cluster"] == cuisine
    mask &= df["score"].fillna(0) >= min_score
    mask &= df["number of reviews"].fillna(0) >= min_revs
    return df[mask]

def refresh(_=None):
    with out:
        clear_output(wait=True)
        df_f = filter_restaurants(restaurants, cuisine_dd.value, min_rating.value, min_reviews_w.value)
        sources = list(source_select.value) if source_select.value else all_sources
        rev_f = reviews[reviews["source"].isin(sources)]
        cat_f = categories_long[categories_long["id"].isin(df_f["id"])]
        print(f"Showing {len(df_f)} restaurants | Sources: {', '.join(sources)}")
        if show_map_cb.value:
            display(make_map(df_f))
        display(make_rating_hist(df_f))
        display(make_category_bar(cat_f, top_n=top_n_slider.value))
        display(make_cross_source_scatter(df_f))
        display(make_sentiment_scatter(rev_f))
        display(make_top_table(df_f))

for w in (cuisine_dd, min_rating, min_reviews_w, source_select, top_n_slider, show_map_cb):
    w.observe(refresh, names="value")

controls = widgets.VBox([
    widgets.HTML("<h4 style='margin:4px 0'>Dashboard Controls</h4>"),
    widgets.HBox([cuisine_dd, show_map_cb]),
    widgets.HBox([min_rating, min_reviews_w]),
    widgets.HBox([top_n_slider, source_select]),
])
display(controls)
refresh()
display(out)

Showing 283 restaurants | Sources: tripadvisor, yellowpages


Output()

## 6. Findings & Notes

- **Score agreement:** Google and TripAdvisor scores are generally correlated, though TripAdvisor ratings tend to be [higher/lower — fill in after running].
- **Top cuisines:** The most represented categories are [fill in after running] — reflecting the city's dining culture.
- **Sentiment alignment:** Reviews with positive sentiment scores reliably correspond to higher star ratings, confirming the sentiment model's validity.
- **Geographic clustering:** Restaurants cluster around [fill in after running] — the city's main dining districts.
- **Data quality:** [X]% of restaurants were successfully geocoded. Missing coordinates are concentrated in [fill in after running].
- **Streamlit app:** The public-facing version of this dashboard is available via `streamlit run streamlit_app.py`. All charts and filters are replicated in the Streamlit interface.

---
*Data scraped by Raphael Altomar. Dashboard built with Plotly, Folium, and ipywidgets.*